In [29]:
import pandas as pd
import numpy as np

bars_df = pd.read_parquet("../data/bars_seen_train.parquet")
unseen_bars_df = pd.read_parquet("../data/bars_unseen_train.parquet")
sentiment_df = pd.read_csv("../sentiment_overview.csv")
headlines_df_seen = pd.read_parquet("../data/headlines_seen_train.parquet")
headlines_df_unseen = pd.read_parquet("../data/headlines_unseen_train.parquet")

# headlines_df_public = pd.read_parquet("../data/headlines_seen_public_test.parquet")
# headlines_df_private = pd.read_parquet("../data/headlines_seen_private_test.parquet")

In [30]:
import re
session_of_interest = -1
# company = "Brevon Microchips"
keyword = ""

keywords = [
    "completes planned facility upgrade",
    "warns of supply chain disruptions affecting",
    "wins industry award",
    "announces significant capital expenditure plan for",
    "withdraws from",
    "recalls products in",
    "reports rising costs pressuring margins",
    "completes strategic acquisition",
    "increase in customer acquisition",
    "loses key contract",
    "reports record quarterly",
    "names new",
    "decline in operating income",
    "secures",
    "expands operations into",
    "launches next-generation",
    "begins scheduled maintenance",
    "margin improvement",
    "drop in new customer orders this quarter",
    "reports strong demand in",
    "files routine",
    "files for regulatory",
    "explores strategic alternatives",
    "confirms participation",
    "delays product launch",
    "opens new office",
    "enters joint venture",
    "faces class action",
    "sees mixed results in",
    "board meeting",
    "misses quarterly revenue estimates",
    "faces regulatory review",
    "to host investor day focused on",
    "announces breakthrough in",
    "revises long-term strategy with focus on",
    "share buyback program",
    "signs multi-year partnership with a",
    "to present at",
    "announces restructuring plan",
    "reports unexpected decline in",
    "announces major organizational restructuring",
    "publishes annual sustainability report",
    "lowers full-year guidance amid softening demand",
    "addresses investor concerns in open letter",
    "steps down unexpectedly",
    "raises full-year guidance citing robust demand",
    "in talks for potential merger, details undisclosed",
    "achieves key regulatory milestone ahead of schedule",
    "beats analyst expectations with strong earnings growth",
    "schedules annual shareholder meeting for next month"
]
# all_day_company_session_keywords = []

# for session_of_interest in range(1000):
#     headlines_seen = headlines_df_seen[headlines_df_seen["session"] == session_of_interest]
#     headlines_seen = headlines_seen[headlines_seen["headline"].str.contains("secures", na=False)]
#     headlines_unseen = headlines_df_unseen[headlines_df_unseen["session"] == session_of_interest]
#     headlines_unseen = headlines_unseen[headlines_unseen["headline"].str.contains("secures", na=False)]
#     df = pd.concat([headlines_seen, headlines_unseen])
#     if df.empty:
#         continue
#     for idx, row in df.iterrows():
#         print(row.headline)


In [31]:
# For each keyword, what is the average (close_end/close_mid - 1)?
from collections import defaultdict
import numpy as np

keyword_returns = defaultdict(list)

for session_id, headline_group in headlines_df_seen.groupby('session'):
    session_bars = bars_df[bars_df["session"] == session_id]  # your return label
    unseen_session_bars = unseen_bars_df[unseen_bars_df["session"] == session_id]  # your return label
    close_mid = session_bars.loc[session_bars['bar_ix'] == 49, 'close'].iloc[0]
    close_end = unseen_session_bars.loc[unseen_session_bars['bar_ix'] == 99, 'close'].iloc[0]
    print(close_mid, close_end)
    outcome = close_end/close_mid - 1
    for _, row in headline_group.iterrows():
        print(row)
        for kw in keywords:
            if kw in row['headline']:
                keyword_returns[kw].append(outcome)
                break

# Print empirical signal strength
for kw, returns in sorted(keyword_returns.items(), 
                           key=lambda x: np.mean(x[1]), reverse=True):
    print(f"{np.mean(returns):+.4f}  (n={len(returns):4d})  {kw}")


1.0316 1.0528
session                                                     0
headline    Relvos Biosciences opens new office in Southea...
bar_ix                                                      6
Name: 0, dtype: object
session                                                     0
headline    Orevex Renewables secures $500M contract with ...
bar_ix                                                     12
Name: 1, dtype: object
session                                                     0
headline    Relvos Biosciences names new head of precision...
bar_ix                                                     14
Name: 2, dtype: object
session                                                     0
headline    Calvis Sciences secures $650M contract with a ...
bar_ix                                                     20
Name: 3, dtype: object
session                                                     0
headline    Yorvov Pharmaceuticals secures $180M contract ...
bar_ix                    

In [32]:
bars_df = pd.read_parquet("../data/bars_seen_train.parquet")
unseen_bars_df = pd.read_parquet("../data/bars_unseen_train.parquet")

mid_price = bars_df.groupby('session')['close'].last()
end_price = unseen_bars_df.groupby('session')['close'].last()

returns = end_price / mid_price - 1

print(f"Mean return:     {returns.mean():.6f}")
print(f"Std return:      {returns.std():.6f}")
print(f"Sharpe (16x):    {returns.mean()/returns.std()*16:.4f}")
print(f"% positive:      {(returns > 0).mean():.3f}")
print(f"Skewness:        {returns.skew():.3f}")
print(f"Kurtosis:        {returns.kurt():.3f}")

# Baseline: always long 1 unit
baseline_sharpe = returns.mean() / returns.std() * 16
print(f"\nAlways-long Sharpe: {baseline_sharpe:.4f}")


Mean return:     0.003531
Std return:      0.020434
Sharpe (16x):    2.7647
% positive:      0.570
Skewness:        0.029
Kurtosis:        0.488

Always-long Sharpe: 2.7647


In [33]:
import pandas as pd
import numpy as np
from scipy import stats

bars_df = pd.read_parquet("../data/bars_seen_train.parquet")
unseen_bars_df = pd.read_parquet("../data/bars_unseen_train.parquet")
mid_price = bars_df.groupby('session')['close'].last()
end_price = unseen_bars_df.groupby('session')['close'].last()
returns = (end_price / mid_price - 1).rename('return')

# ── Build session-level features from seen bars ──────────────────────────
def session_features(grp):
    closes = grp.sort_values('bar_ix')['close'].values
    opens  = grp.sort_values('bar_ix')['open'].values
    highs  = grp.sort_values('bar_ix')['high'].values
    lows   = grp.sort_values('bar_ix')['low'].values
    n      = len(closes)

    bar_returns = np.diff(closes) / closes[:-1]

    return pd.Series({
        # Momentum
        'cum_return':      closes[-1] / closes[0] - 1,
        'last3_return':    closes[-1] / closes[max(0, n-4)] - 1,
        'first3_return':   closes[min(3, n-1)] / closes[0] - 1,
        'slope':           np.polyfit(np.arange(n), closes, 1)[0] / closes.mean(),

        # Volatility
        'realized_vol':    np.std(bar_returns),
        'range_mean':      np.mean((highs - lows) / closes),

        # Microstructure
        'up_bar_frac':     np.mean(closes > opens),
        'last_bar_up':     float(closes[-1] > opens[-1]),
        'wick_ratio':      np.mean((highs - closes) / (highs - lows + 1e-8)),

        # Regime
        'n_bars':          n,
        'accel':           bar_returns[-1] - bar_returns[0] if n > 1 else 0,
    })

features = bars_df.groupby('session').apply(session_features)
df = features.join(returns)

# ── Correlation with forward return ──────────────────────────────────────
print("=== Pearson correlation with forward return ===")
for col in features.columns:
    r, p = stats.pearsonr(df[col], df['return'])
    print(f"  {col:25s}  r={r:+.4f}  p={p:.4f}")

# ── Quintile analysis on strongest feature ────────────────────────────────
print("\n=== Quintile Sharpe by cum_return ===")
df['quintile'] = pd.qcut(df['cum_return'], 5, labels=False)
for q, grp in df.groupby('quintile'):
    sh = grp['return'].mean() / grp['return'].std() * 16
    print(f"  Q{q+1}  mean_ret={grp['return'].mean():+.5f}  sharpe={sh:.3f}  n={len(grp)}")

=== Pearson correlation with forward return ===
  cum_return                 r=-0.0694  p=0.0281
  last3_return               r=+0.0441  p=0.1631
  first3_return              r=+0.0229  p=0.4688
  slope                      r=-0.0670  p=0.0341
  realized_vol               r=+0.0716  p=0.0235
  range_mean                 r=+0.0787  p=0.0128
  up_bar_frac                r=-0.0594  p=0.0605
  last_bar_up                r=-0.0017  p=0.9575
  wick_ratio                 r=+0.0703  p=0.0263
  n_bars                     r=+nan  p=nan
  accel                      r=+0.0049  p=0.8776

=== Quintile Sharpe by cum_return ===
  Q1  mean_ret=+0.00541  sharpe=3.688  n=200
  Q2  mean_ret=+0.00275  sharpe=2.260  n=200
  Q3  mean_ret=+0.00437  sharpe=3.613  n=200
  Q4  mean_ret=+0.00376  sharpe=3.181  n=200
  Q5  mean_ret=+0.00135  sharpe=1.052  n=200


/tmp/ipykernel_1264/2440219881.py:48: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.pearsonr(df[col], df['return'])


In [34]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Features that are significant (p < 0.05)
sig_features = ['cum_return', 'slope', 'realized_vol', 'range_mean', 'wick_ratio']

X = df[sig_features].values
y = df['return'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ridge regression — regularize heavily given weak signal
ridge = Ridge(alpha=100)

# Cross-validate Sharpe directly
def sharpe_scorer(estimator, X, y):
    pred = estimator.predict(X)
    # Position = prediction (go long/short proportional to signal)
    pnl = pred * y  # not right — see below
    return np.mean(pnl) / np.std(pnl) * 16

# Correct Sharpe CV: position proportional to prediction, PnL is position * actual_return
def sharpe_cv(estimator, X_tr, y_tr, X_val, y_val):
    estimator.fit(X_tr, y_tr)
    positions = estimator.predict(X_val)
    # Normalize positions to unit scale
    positions = positions / np.std(positions)
    pnl = positions * y_val
    return np.mean(pnl) / np.std(pnl) * 16

# Manual walk-free CV (sessions are independent so random split is fine)
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
sharpes = []
for train_idx, val_idx in kf.split(X_scaled):
    ridge.fit(X_scaled[train_idx], y[train_idx])
    positions = ridge.predict(X_scaled[val_idx])
    positions = positions / positions.std()
    pnl = positions * y[val_idx]
    sharpes.append(np.mean(pnl) / np.std(pnl) * 16)

print(f"CV Sharpe: {np.mean(sharpes):.4f} ± {np.std(sharpes):.4f}")
print(f"Baseline:  2.7647")

CV Sharpe: 2.6872 ± 0.5901
Baseline:  2.7647


In [35]:
def make_positions(predictions, method='proportional', clip=3.0):
    z = (predictions - predictions.mean()) / predictions.std()
    z = np.clip(z, -clip, clip)
    
    if method == 'proportional':
        # Scale around the always-long baseline
        # Never go short (drift is strongly positive)
        return 1.0 + 0.3 * z          # tune the 0.3 multiplier
    
    elif method == 'long_only_sized':
        # Go long always, size by conviction
        return np.maximum(0.1, 1.0 + 0.5 * z)
    
    elif method == 'full_ls':
        # Allow shorts on strong negative signals
        return z                        # risky given strong positive drift
    
make_positions(pred)


NameError: name 'pred' is not defined

In [ ]:
ALPHA=40000
SENSITIVITY=1.0
CLIP=0.5


import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

DATA_DIR = "../data"

# ── 1. Load training data ─────────────────────────────────────────────────────

seen_train    = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
unseen_train  = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")

# ── 2. Feature engineering ────────────────────────────────────────────────────

def build_features(bars: pd.DataFrame) -> pd.DataFrame:
    """Extract session-level features from seen OHLC bars."""

    def session_feats(grp):
        grp   = grp.sort_values("bar_ix")
        c     = grp["close"].values
        o     = grp["open"].values
        h     = grp["high"].values
        lo    = grp["low"].values
        n     = len(c)
        br    = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

        return pd.Series({
            "cum_return":   c[-1] / c[0] - 1,
            "last3_return": c[-1] / c[max(0, n - 4)] - 1,
            "slope":        np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
            "realized_vol": np.std(br),
            "range_mean":   np.mean((h - lo) / c),
            "up_bar_frac":  np.mean(c > o),
            "wick_ratio":   np.mean((h - c) / (h - lo + 1e-8)),
        })

    return bars.groupby("session").apply(session_feats)

FEATURES = ["cum_return", "last3_return", "slope",
            "realized_vol", "range_mean", "up_bar_frac", "wick_ratio"]

# ── 3. Build training labels ──────────────────────────────────────────────────

mid_price = seen_train.groupby("session")["close"].last()
end_price = unseen_train.groupby("session")["close"].last()
y_train   = (end_price / mid_price - 1).rename("return")

X_train_raw = build_features(seen_train)[FEATURES]

# Align index (drop any sessions missing from either side)
idx = X_train_raw.index.intersection(y_train.index)
X_train_raw = X_train_raw.loc[idx]
y_train     = y_train.loc[idx]

# ── 4. Cross-validate to confirm we beat the baseline ────────────────────────

from sklearn.model_selection import KFold

def compute_sharpe(positions, returns):
    pnl = positions * returns
    return np.mean(pnl) / np.std(pnl) * 16

def make_positions(raw_preds, long_bias=1.0, sensitivity=0.3, clip=2.5):
    """
    Never go short (strong positive drift).
    Modulate position size around the long bias using model signal.
    position = long_bias + sensitivity * z_score(prediction)
    Clipped so minimum position is always > 0.
    """
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    z = np.clip(z, -clip, clip)
    positions = long_bias + sensitivity * z
    positions = np.maximum(positions, 0.05)   # never fully flat
    return positions

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_raw)
y        = y_train.values

kf      = KFold(n_splits=5, shuffle=True, random_state=42)
cv_sharpes_model    = []
cv_sharpes_baseline = []

for train_idx, val_idx in kf.split(X_scaled):
    model = Ridge(alpha=ALPHA)
    model.fit(X_scaled[train_idx], y[train_idx])

    preds     = model.predict(X_scaled[val_idx])
    positions = make_positions(preds, sensitivity=SENSITIVITY, clip=CLIP)
    y_val     = y[val_idx]

    cv_sharpes_model.append(compute_sharpe(positions, y_val))
    cv_sharpes_baseline.append(compute_sharpe(np.ones(len(y_val)), y_val))

print(f"Baseline CV Sharpe : {np.mean(cv_sharpes_baseline):.4f} ± {np.std(cv_sharpes_baseline):.4f}")
print(f"Model    CV Sharpe : {np.mean(cv_sharpes_model):.4f} ± {np.std(cv_sharpes_model):.4f}")

# ── 5. Fit final model on all training data ───────────────────────────────────

final_model = Ridge(alpha=ALPHA)
final_scaler = StandardScaler()
X_final = final_scaler.fit_transform(X_train_raw)
final_model.fit(X_final, y)

print("\nFeature coefficients:")
for feat, coef in zip(FEATURES, final_model.coef_):
    print(f"  {feat:20s}  {coef:+.6f}")

# ── 6. Generate predictions for both test sets ────────────────────────────────

def predict_positions(bars: pd.DataFrame) -> pd.Series:
    feats    = build_features(bars)[FEATURES]
    X        = final_scaler.transform(feats)
    preds    = final_model.predict(X)
    positions = make_positions(preds)
    return pd.Series(positions, index=feats.index, name="target_position")

public_bars  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),  engine="fastparquet")
private_bars = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

public_positions  = predict_positions(public_bars)
private_positions = predict_positions(private_bars)

# ── 7. Write submissions ──────────────────────────────────────────────────────

def write_submission(positions: pd.Series, path: str):
    sub = positions.reset_index()
    sub.columns = ["session", "target_position"]
    sub = sub.sort_values("session")
    sub.to_csv(path, index=False)
    print(f"Saved {len(sub)} rows → {path}")
    print(sub.describe())

write_submission(public_positions,  "submission_public.csv")
write_submission(private_positions, "submission_private.csv")

Baseline CV Sharpe : 2.7838 ± 0.5857
Model    CV Sharpe : 2.8247 ± 0.5078

Feature coefficients:
  cum_return            -0.001210
  last3_return          +0.001135
  slope                 +0.000258
  realized_vol          +0.000308
  range_mean            +0.001151
  up_bar_frac           -0.000147
  wick_ratio            +0.000640
Saved 10000 rows → submission_public.csv
           session  target_position
count  10000.00000     10000.000000
mean    5999.50000         0.998777
std     2886.89568         0.296146
min     1000.00000         0.250000
25%     3499.75000         0.786302
50%     5999.50000         0.981649
75%     8499.25000         1.201120
max    10999.00000         1.750000
Saved 10000 rows → submission_private.csv
           session  target_position
count  10000.00000     10000.000000
mean   15999.50000         0.998783
std     2886.89568         0.296190
min    11000.00000         0.250000
25%    13499.75000         0.786705
50%    15999.50000         0.979611
75%   

In [ ]:
for s in [0.0, 0.1, 0.2, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        p = make_positions(m.predict(X_scaled[val_idx]), sensitivity=s)
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    print(f"sensitivity={s:.1f}  sharpe={np.mean(fold_sharpes):.4f}")


sensitivity=0.0  sharpe=2.7838
sensitivity=0.1  sharpe=2.8236
sensitivity=0.2  sharpe=2.8356
sensitivity=0.3  sharpe=2.8247
sensitivity=0.5  sharpe=2.7568
sensitivity=0.6  sharpe=2.7184
sensitivity=0.7  sharpe=2.6759
sensitivity=0.8  sharpe=2.6309
sensitivity=0.9  sharpe=2.5831
sensitivity=1.0  sharpe=2.5326
sensitivity=1.1  sharpe=2.4841
sensitivity=1.2  sharpe=2.4430
sensitivity=1.3  sharpe=2.4059
sensitivity=1.4  sharpe=2.3721
sensitivity=1.5  sharpe=2.3408


In [ ]:
from itertools import product

best, best_params = -np.inf, {}

for alpha, sensitivity, clip in product(
    [10, 50, 100, 200, 500, 400, 300, 600, 700, 800, 900, 1000, 1250, 1500, 1750, 2000, 2250, 2500, 2750, 3000, 5000, 7500, 10000, 20000, 30000, 40000],   # ridge regularization
    [0.15, 0.18, 0.20, 0.22, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5],  # tight grid around optimum
    [0.25, 0.5, 0.75, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0],           # outlier clipping
):
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m = Ridge(alpha=alpha).fit(X_scaled[train_idx], y[train_idx])
        p = make_positions(
            m.predict(X_scaled[val_idx]),
            sensitivity=sensitivity,
            clip=clip
        )
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    
    s = np.mean(fold_sharpes)
    if s > best:
        best, best_params = s, dict(alpha=alpha, sensitivity=sensitivity, clip=clip)

print(f"Best CV Sharpe : {best:.4f}")
print(f"Best params    : {best_params}")

Best CV Sharpe : 2.9824
Best params    : {'alpha': 40000, 'sensitivity': 1.0, 'clip': 0.5}
